# CrewAI Hiring Pipeline Crew

**Project:** Multi-Agent Resume Screening & Interview Preparation Pipeline  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent hiring pipeline crew** that processes candidate applications end-to-end — from resume screening through interview question generation, candidate summarization, and bias review.

### The Agent Roster

| # | Agent | Role | Deliverable |
|---|-------|------|------------|
| 1 | **Resume Screener** | Skills & Experience Matcher | Structured screening scorecard — requirements fit, gaps, flags |
| 2 | **Question Generator** | Interview Preparation Specialist | Tailored interview questions — technical, behavioral, gap-probing |
| 3 | **Candidate Summarizer** | Hiring Committee Briefer | One-page candidate brief — strengths, concerns, hiring recommendation |
| 4 | **Bias-Check Reviewer** | Fairness & Compliance Auditor | Bias audit — flags biased language, proxy discrimination, compliance issues |

### Pipeline Flow

```
Job Description + Resume
          │
          ▼
┌─────────────────────┐
│   Resume Screener   │  SCREEN — "Does this candidate meet the requirements?"
└──────────┬──────────┘
           │ Screening scorecard (skills match, gaps, flags)
           ▼
┌─────────────────────┐
│ Question Generator  │  PREPARE — "What should we ask in the interview?"
└──────────┬──────────┘
           │ Tailored question set (technical, behavioral, gap-probing)
           ▼
┌─────────────────────┐
│Candidate Summarizer │  BRIEF — "What does the hiring committee need to know?"
└──────────┬──────────┘
           │ One-page candidate brief with recommendation
           ▼
┌─────────────────────┐
│ Bias-Check Reviewer │  AUDIT — "Is this process fair and compliant?"
└─────────────────────┘
           │
           ▼
  Audited hiring package

```

---

## ⚠️ Limitations & Ethical Considerations

**This notebook is an educational demonstration of multi-agent AI patterns, NOT a production hiring tool.**

Before studying the code, read these limitations carefully:

### Critical Limitations of AI in Hiring

| Limitation | Why It Matters | Mitigation |
|-----------|---------------|-----------|
| **LLMs hallucinate qualifications** | The model may infer skills not present on the resume or miss stated ones | Always verify screening against the actual resume text |
| **Training data bias** | LLMs trained on internet text reflect historical hiring biases (gender, race, age, disability, socioeconomic) | A bias-check agent helps but cannot eliminate embedded biases |
| **No legal compliance guarantee** | AI cannot ensure compliance with EEOC, ADA, GDPR, or local employment law | Human legal review is mandatory for any production system |
| **Proxy discrimination** | Factors like university name, zip code, or hobbies can serve as proxies for protected characteristics | The bias reviewer flags proxies, but may not catch all of them |
| **Lack of context** | LLMs cannot verify employment history, assess cultural fit authentically, or detect resume fraud | AI screening supplements — never replaces — human judgment |
| **Overconfidence in scores** | Numerical scores create an illusion of objectivity that masks subjective language interpretation | Treat all scores as conversation starters, not decisions |

### What This Demo Does Well vs. What It Cannot Do

| This Demo Shows | This Demo Cannot |
|----------------|-----------------|
| How to structure multi-agent pipelines for sequential document processing | Replace human hiring judgment |
| How to add a bias-check agent as a safety layer | Guarantee bias-free outcomes |
| How agent outputs chain together (screening → questions → summary → audit) | Validate resume accuracy |
| How to design agents with fairness-aware backstories | Comply with employment law in your jurisdiction |
| Educational patterns for responsible AI design | Make legally defensible hiring decisions |

**Bottom line:** Use this notebook to learn multi-agent patterns. Do NOT use the outputs to make real hiring decisions without qualified human review and legal counsel.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

All agents share the same local Ollama model. The bias-check reviewer uses the same model as the other agents — it cannot detect biases the model itself has internalized. This is a fundamental limitation discussed in Section 11.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### Designing for the Hiring Domain

Hiring pipelines have unique design requirements compared to other multi-agent systems:

| Requirement | Why | How We Address It |
|------------|-----|------------------|
| **Transparency** | Candidates and companies need to understand decisions | Screening uses explicit criteria matching, not opaque scores |
| **Consistency** | Every candidate should be evaluated against the same rubric | Standard scorecard structure across all candidates |
| **Auditability** | Hiring decisions may face legal scrutiny | Bias-check agent produces an audit trail |
| **Humility** | AI should flag uncertainty, not project false confidence | Agents are instructed to distinguish "confirmed" from "inferred" |

**Limitation note:** Despite these design choices, an LLM-based system cannot guarantee fairness. The bias-check agent is an additional safety layer, not a substitute for human oversight.

### 3.1 Resume Screener Agent

**Pipeline role:** First evaluator — matches the candidate's stated experience against job requirements.

**Limitation-aware design:** The backstory explicitly instructs the agent to distinguish between "confirmed on resume" and "inferred/assumed" qualifications. LLMs tend to hallucinate skills (inferring Python experience from a data science role, for example) — this instruction mitigates but does not eliminate that tendency.

In [ ]:
resume_screener = Agent(
    role="Resume Screening Specialist & Requirements Matcher",
    goal=(
        "Screen the candidate's resume against the job description requirements. "
        "Produce a structured scorecard that rates the candidate on each requirement, "
        "identifies skill gaps, flags strengths, and provides an overall fit assessment. "
        "Distinguish between qualifications CONFIRMED on the resume vs. INFERRED."
    ),
    backstory=(
        "You are a talent acquisition specialist with 10 years of experience screening "
        "technical resumes at mid-to-large technology companies. You have screened 15,000+ "
        "resumes and developed a structured screening methodology: (1) Requirements Matching "
        "— for each job requirement, check: CONFIRMED (explicitly stated on resume with "
        "evidence), PARTIAL (related but not exact match), INFERRED (likely based on role "
        "titles but not explicitly stated — mark clearly), NOT FOUND (no evidence on resume). "
        "(2) Experience Assessment — total years, relevance of roles, progression trajectory, "
        "recency of relevant experience. (3) Education Check — degree match, institution "
        "level, relevant coursework or certifications. (4) Red Flags — employment gaps > 12 "
        "months (note but do NOT penalize without context), frequent job changes < 1 year, "
        "role title inflation, vague descriptions. (5) Overall Fit Score — Strong Fit, "
        "Moderate Fit, Weak Fit, No Fit. CRITICAL: You MUST distinguish between what the "
        "resume explicitly states and what you are inferring. Never present an inference as "
        "a confirmed qualification. When uncertain, say 'Not confirmed on resume.'"
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {resume_screener.role}")

### 3.2 Question Generator Agent

**Pipeline role:** Interview preparation — creates tailored questions based on the screening results.

**Limitation-aware design:** The backstory mandates a mix of structured and behavioral questions to reduce the risk of biased evaluation. It also explicitly avoids questions about protected characteristics (age, family status, origin, religion, disability).

In [ ]:
question_generator = Agent(
    role="Technical Interview Question Designer & Gap Probe Specialist",
    goal=(
        "Design a tailored set of 12-15 interview questions for this candidate "
        "based on the screening results. Questions must probe skill gaps, validate "
        "claimed experience, and assess behavioral fit. Include scoring rubrics for "
        "each question. Never ask questions about protected characteristics."
    ),
    backstory=(
        "You are a senior technical interviewer and interview design consultant with "
        "8 years of experience building structured interview frameworks for tech companies. "
        "You design questions across four categories: (1) Technical Validation — questions "
        "that verify the candidate actually has the skills listed on their resume. These use "
        "specific scenarios, not trivia. (2) Gap Probing — questions that explore areas where "
        "the screening found PARTIAL or NOT FOUND matches. These are fair explorations, not "
        "gotchas. (3) Behavioral/Situational — STAR-format questions (Situation, Task, Action, "
        "Result) about teamwork, conflict, learning, and leadership. (4) Role-Specific Scenarios "
        "— realistic work scenarios the candidate would face in this role. For each question you "
        "provide: the question text, category, what it assesses, what a strong answer looks like, "
        "and what a weak answer looks like. FORBIDDEN TOPICS: age, marital/family status, "
        "national origin, religion, disability, pregnancy, genetic information, sexual "
        "orientation, political affiliation, arrest history. If a gap relates to a protected "
        "characteristic, reframe the question to focus on job-relevant skills only."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {question_generator.role}")

### 3.3 Candidate Summarizer Agent

**Pipeline role:** Hiring committee briefer — distills the screening and questions into a concise, decision-ready summary.

**Limitation-aware design:** The backstory requires the summary to explicitly state confidence levels for each assessment and to flag areas where the AI cannot make a reliable judgment (cultural fit, motivation, communication skills).

In [ ]:
candidate_summarizer = Agent(
    role="Candidate Briefing Specialist & Hiring Committee Advisor",
    goal=(
        "Produce a one-page candidate brief for the hiring committee that synthesizes "
        "the screening results and interview preparation. Include strengths, concerns, "
        "key questions to ask, and a preliminary recommendation with confidence level. "
        "Flag areas where AI assessment is unreliable and human judgment is needed."
    ),
    backstory=(
        "You are a hiring operations lead with 7 years of experience preparing candidate "
        "briefs for hiring committees at companies with structured hiring processes (Google, "
        "Stripe-style hiring). You write the document that 3-5 interviewers read before "
        "meeting the candidate. Your brief format: (1) Candidate Snapshot — 3-sentence "
        "summary of who this person is professionally, (2) Strengths — top 3-5 reasons "
        "to hire, each with evidence from the resume, (3) Concerns — top 3-5 questions or "
        "risks, each with specific gaps to probe, (4) Key Interview Areas — the 3 most "
        "important things the interview should validate, (5) Recommendation — Advance / "
        "Advance with Reservations / Do Not Advance, with confidence: High (strong evidence) "
        "/ Medium (some evidence, needs validation) / Low (insufficient data to assess). "
        "CRITICAL: You MUST include an 'AI Assessment Limitations' section listing things "
        "this automated process CANNOT reliably assess: communication skills, cultural fit, "
        "motivation authenticity, interpersonal dynamics, reference quality, and work sample "
        "quality. These require human evaluation in the interview."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {candidate_summarizer.role}")

### 3.4 Bias-Check Reviewer Agent

**Pipeline role:** Fairness auditor — reviews ALL prior outputs for bias indicators.

**Limitation-aware design:** This is the most limitation-critical agent. Its backstory acknowledges that:
1. The same LLM that may have produced biased outputs is checking for bias
2. It can detect explicit bias markers but may miss subtle or systemic bias
3. It flags issues for human review rather than claiming to guarantee fairness

**Why bias review must be last:** The reviewer needs to see the entire pipeline's output — screening, questions, and summary — to detect bias that may have accumulated across agents.

In [ ]:
bias_reviewer = Agent(
    role="Hiring Fairness Auditor & Bias Detection Specialist",
    goal=(
        "Audit the complete hiring package (screening, questions, summary) for "
        "potential bias, proxy discrimination, and compliance risks. Flag any "
        "language, criteria, or reasoning that could disadvantage candidates "
        "based on protected characteristics. Produce an audit report with "
        "specific findings and recommended fixes."
    ),
    backstory=(
        "You are an I/O psychologist and employment fairness consultant with 10 years of "
        "experience auditing hiring processes for Fortune 500 companies. You have reviewed "
        "1,000+ hiring packages and testified as an expert witness in employment discrimination "
        "cases. Your audit framework covers: (1) Language Bias — flags gendered language "
        "(e.g., 'aggressive' skews male, 'nurturing' skews female), age-coded terms ('digital "
        "native,' 'energetic'), disability-exclusionary language ('must be able to stand'), and "
        "socioeconomic proxies ('top-tier university,' 'prestigious firm'). (2) Proxy "
        "Discrimination — identifies criteria that correlate with protected characteristics: "
        "zip code (race/income), graduation year (age), university name (race/class), name or "
        "cultural markers, club/sports/hobbies (gender/class). (3) Requirement Inflation — "
        "flags requirements that exceed what the job actually needs (e.g., requiring a Master's "
        "for a role that doesn't need one), which disproportionately excludes underrepresented "
        "groups. (4) Evaluation Consistency — checks whether the same criteria were applied "
        "equally. (5) Compliance Flags — EEOC, ADA, ADEA, Title VII, and GDPR (if applicable) "
        "risk areas. For each finding, you rate severity: Critical (likely illegal / discriminatory), "
        "Major (bias risk, needs revision), Minor (best practice improvement). You conclude with "
        "a PASS / CONDITIONAL PASS / FAIL verdict. IMPORTANT LIMITATION: You are an AI and "
        "CANNOT guarantee bias-free outcomes. Your audit catches explicit bias markers but may "
        "miss subtle or systemic bias patterns. Your findings are flags for human review, NOT "
        "legal compliance certification."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {bias_reviewer.role}")

### Agent Roster Summary

In [ ]:
agents = [resume_screener, question_generator, candidate_summarizer, bias_reviewer]

print("=" * 70)
print("HIRING PIPELINE CREW — AGENT ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Goal: {agent.goal[:90]}...")
    print(f"    Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)
print("\nAll agents instructed to flag limitations and defer to human judgment.")

## 4. Define Tasks

### 4.1 Configure the Job Description & Resume

Both inputs are synthetic examples — no real personal data is used.

In [ ]:
# =====================================================
# SYNTHETIC JOB DESCRIPTION
# =====================================================
JOB_DESCRIPTION = {
    "title": "Senior Machine Learning Engineer",
    "company": "DataStream Analytics (Series B, 120 employees)",
    "team": "ML Platform Team (6 engineers)",
    "location": "Remote (US timezone overlap required)",
    "requirements": {
        "must_have": [
            "3+ years building production ML systems",
            "Strong Python (scikit-learn, PyTorch or TensorFlow)",
            "Experience with ML pipelines (Kubeflow, Airflow, or MLflow)",
            "SQL proficiency for data extraction and feature engineering",
            "Experience with cloud platforms (AWS, GCP, or Azure)",
        ],
        "nice_to_have": [
            "Experience with LLMs or NLP models",
            "Familiarity with real-time inference systems",
            "Contributions to open-source ML projects",
            "Experience mentoring junior engineers",
        ],
    },
    "salary_range": "$160K-$200K + equity",
}

print("JOB DESCRIPTION")
print("=" * 60)
print(f"  Title:    {JOB_DESCRIPTION['title']}")
print(f"  Company:  {JOB_DESCRIPTION['company']}")
print(f"  Team:     {JOB_DESCRIPTION['team']}")
print(f"  Location: {JOB_DESCRIPTION['location']}")
print(f"  Salary:   {JOB_DESCRIPTION['salary_range']}")
print(f"\n  Must-Have Requirements:")
for r in JOB_DESCRIPTION['requirements']['must_have']:
    print(f"    ✓ {r}")
print(f"\n  Nice-to-Have:")
for r in JOB_DESCRIPTION['requirements']['nice_to_have']:
    print(f"    ○ {r}")

In [ ]:
# =====================================================
# SYNTHETIC RESUME (fictional candidate)
# =====================================================
RESUME = """
ALEX MORGAN
alex.morgan@email.com | github.com/alexmorgan-ml

SUMMARY
ML engineer with 4 years of experience building data pipelines and deploying
machine learning models. Passionate about using AI to solve real-world problems.

EXPERIENCE

Machine Learning Engineer — TechNova Corp (2022–Present, 2.5 years)
- Built and deployed 3 production ML models for customer churn prediction and
  recommendation systems using Python, scikit-learn, and PyTorch
- Designed feature engineering pipelines using Apache Airflow and BigQuery
- Reduced model inference latency by 40% by optimizing serving infrastructure on GCP
- Collaborated with product team to define success metrics and A/B testing frameworks
- Mentored 2 junior data scientists on ML best practices

Data Scientist — AnalytiCore Solutions (2020–2022, 2 years)
- Developed NLP classification models for document categorization using spaCy and BERT
- Built automated reporting dashboards using Python and SQL
- Managed data extraction pipelines from PostgreSQL and Snowflake
- Presented ML findings to stakeholders in weekly cross-functional meetings

EDUCATION
B.S. Computer Science — State University (2020)
Online ML Specialization — Coursera (Andrew Ng, completed 2021)

SKILLS
Python, PyTorch, scikit-learn, SQL, Apache Airflow, GCP (BigQuery, Vertex AI),
Docker, Git, spaCy, Hugging Face Transformers, pandas, numpy

CERTIFICATIONS
- Google Cloud Professional Machine Learning Engineer (2023)
- AWS Solutions Architect Associate (2022)

PROJECTS
- Open-source contributor to Hugging Face Datasets library (12 merged PRs)
- Personal project: real-time sentiment analysis API using FastAPI + distilBERT
"""

print("CANDIDATE RESUME")
print("=" * 60)
print(RESUME)

In [ ]:
# Build the combined context
jd_text = (
    f"=== JOB DESCRIPTION ===\n"
    f"Title: {JOB_DESCRIPTION['title']}\n"
    f"Company: {JOB_DESCRIPTION['company']}\n"
    f"Team: {JOB_DESCRIPTION['team']}\n"
    f"Location: {JOB_DESCRIPTION['location']}\n\n"
    f"MUST-HAVE Requirements:\n"
)
for r in JOB_DESCRIPTION['requirements']['must_have']:
    jd_text += f"  - {r}\n"
jd_text += f"\nNICE-TO-HAVE:\n"
for r in JOB_DESCRIPTION['requirements']['nice_to_have']:
    jd_text += f"  - {r}\n"

hiring_context = f"{jd_text}\n=== CANDIDATE RESUME ===\n{RESUME}"
print(f"Context prepared: {len(hiring_context):,} chars")

### 4.2 Task 1 — Resume Screening

**Pipeline phase:** SCREEN — match resume qualifications against job requirements.

**Limitation note:** The screener may infer skills not explicitly stated on the resume. The task description requires marking inferences as such.

In [ ]:
task_screen = Task(
    description=(
        f"Screen this candidate for the following role:\n\n"
        f"{hiring_context}\n\n"
        "Produce a STRUCTURED SCREENING SCORECARD:\n\n"
        "1. **Requirements Match Table**:\n"
        "   For EACH must-have requirement:\n"
        "   - Requirement text\n"
        "   - Status: CONFIRMED / PARTIAL / INFERRED / NOT FOUND\n"
        "   - Evidence: exact quote or reference from resume\n"
        "   - If INFERRED: state your inference and why it's uncertain\n\n"
        "2. **Nice-to-Have Match Table** (same format)\n\n"
        "3. **Experience Assessment**:\n"
        "   - Total years of relevant experience\n"
        "   - Career trajectory (growth, lateral, decline)\n"
        "   - Recency of relevant skills\n\n"
        "4. **Education & Certifications Match**\n\n"
        "5. **Strengths** (top 3 with evidence)\n\n"
        "6. **Concerns / Gaps** (top 3 with specific missing qualifications)\n\n"
        "7. **Red Flags** (employment gaps, inconsistencies, or none)\n\n"
        "8. **Overall Fit**: Strong Fit / Moderate Fit / Weak Fit / No Fit\n"
        "   With confidence level: High / Medium / Low\n\n"
        "CRITICAL: For each qualification, clearly mark whether the evidence is "
        "CONFIRMED (explicitly on resume) or INFERRED (you assumed it). Never "
        "treat an inference as a confirmed qualification.\n"
    ),
    expected_output=(
        "A screening scorecard with requirements match table (CONFIRMED/PARTIAL/"
        "INFERRED/NOT FOUND for each requirement), experience assessment, strengths, "
        "concerns, red flags, and overall fit rating with confidence level."
    ),
    agent=resume_screener,
)

print(f"Task created: Resume Screening -> {task_screen.agent.role}")

### 4.3 Task 2 — Interview Question Generation

**Pipeline phase:** PREPARE — design questions tailored to this specific candidate.

**Limitation note:** Questions are based on the screening scorecard. If the screening missed or misidentified a gap, the questions won't probe the right areas. This cascading dependency is a structural limitation of sequential pipelines.

In [ ]:
task_questions = Task(
    description=(
        "Using the screening scorecard, design a tailored interview question set.\n\n"
        f"Role: {JOB_DESCRIPTION['title']} at {JOB_DESCRIPTION['company']}\n\n"
        "Design 12-15 questions across four categories:\n\n"
        "1. **Technical Validation** (4-5 questions):\n"
        "   - Verify skills marked as CONFIRMED — go deeper than resume claims\n"
        "   - Use scenario-based questions, not trivia\n"
        "   - Example: 'You mentioned reducing inference latency by 40%%. Walk me "
        "     through your approach.'\n\n"
        "2. **Gap Probing** (3-4 questions):\n"
        "   - Explore areas marked PARTIAL or NOT FOUND in the screening\n"
        "   - Frame as fair explorations, not gotchas\n"
        "   - Give the candidate a chance to demonstrate related experience\n\n"
        "3. **Behavioral / STAR** (3-4 questions):\n"
        "   - Teamwork, conflict resolution, learning from failure, mentoring\n"
        "   - Use STAR format: Situation, Task, Action, Result\n\n"
        "4. **Role-Specific Scenarios** (2-3 questions):\n"
        "   - Realistic situations they would face in this ML platform role\n\n"
        "For EACH question provide:\n"
        "- Question text\n"
        "- Category (Technical / Gap Probe / Behavioral / Scenario)\n"
        "- What it assesses\n"
        "- What a STRONG answer includes\n"
        "- What a WEAK answer looks like\n"
        "- Time allocation (minutes)\n\n"
        "FORBIDDEN: Questions about age, family status, national origin, religion, "
        "disability, pregnancy, genetic information, sexual orientation, political "
        "affiliation, or arrest history.\n"
    ),
    expected_output=(
        "A set of 12-15 interview questions across 4 categories (technical, gap "
        "probe, behavioral, scenario), each with assessment criteria, strong/weak "
        "answer guides, and time allocation."
    ),
    agent=question_generator,
)

print(f"Task created: Interview Questions -> {task_questions.agent.role}")

### 4.4 Task 3 — Candidate Summary

**Pipeline phase:** BRIEF — synthesize everything into a hiring committee document.

**Limitation note:** The summary must include an explicit "AI Assessment Limitations" section acknowledging what the automated pipeline cannot reliably evaluate.

In [ ]:
task_summary = Task(
    description=(
        "Produce a one-page candidate brief for the hiring committee.\n\n"
        f"Role: {JOB_DESCRIPTION['title']}\n"
        f"Candidate: Alex Morgan\n\n"
        "Your brief MUST include:\n\n"
        "1. **Candidate Snapshot** (3 sentences):\n"
        "   Who is this person? Key experience. Why they might be a fit.\n\n"
        "2. **Top 5 Strengths** (with evidence from resume + screening)\n\n"
        "3. **Top 3-5 Concerns** (with specific gaps to probe in interview)\n\n"
        "4. **Key Interview Focus Areas**:\n"
        "   The 3 most important things the interview MUST validate\n\n"
        "5. **Preliminary Recommendation**:\n"
        "   Advance / Advance with Reservations / Do Not Advance\n"
        "   Confidence: High / Medium / Low\n"
        "   Reasoning: 2-3 sentences\n\n"
        "6. **AI Assessment Limitations** (REQUIRED section):\n"
        "   Explicitly list what this automated assessment CANNOT reliably evaluate:\n"
        "   - Communication and presentation skills\n"
        "   - Cultural fit and team dynamics\n"
        "   - Motivation and career goals authenticity\n"
        "   - Code quality and problem-solving approach\n"
        "   - Reference quality and work history verification\n"
        "   - Soft skills and leadership presence\n"
        "   State: 'These areas require human evaluation during the interview.'\n\n"
        "FORMAT: Skimmable in 2 minutes. Headers, bullets, bold key findings.\n"
    ),
    expected_output=(
        "A one-page candidate brief with snapshot, strengths, concerns, interview "
        "focus areas, recommendation with confidence level, and an explicit AI "
        "Assessment Limitations section."
    ),
    agent=candidate_summarizer,
)

print(f"Task created: Candidate Summary -> {task_summary.agent.role}")

### 4.5 Task 4 — Bias-Check Review

**Pipeline phase:** AUDIT — review the entire pipeline for bias.

**Limitation note:** This agent uses the SAME LLM that produced the prior outputs. It can detect explicit bias markers (gendered language, age-coded terms, socioeconomic proxies) but may not detect biases the model itself has internalized. This creates a fundamental "auditing yourself" problem discussed in Section 11.

In [ ]:
task_bias = Task(
    description=(
        "Audit the complete hiring package for bias, proxy discrimination, and "
        "compliance risks.\n\n"
        f"Role: {JOB_DESCRIPTION['title']}\n"
        "Candidate: Alex Morgan\n\n"
        "You have received:\n"
        "  1. Screening scorecard\n"
        "  2. Interview questions\n"
        "  3. Candidate summary with recommendation\n\n"
        "Audit across 5 dimensions:\n\n"
        "1. **Language Bias Scan**:\n"
        "   - Gendered language in evaluation or questions\n"
        "   - Age-coded terms ('energetic,' 'digital native,' 'seasoned')\n"
        "   - Disability-exclusionary language\n"
        "   - Socioeconomic proxies ('prestigious university,' 'pedigree')\n\n"
        "2. **Proxy Discrimination Check**:\n"
        "   - University name used as quality signal (correlates with race/class)\n"
        "   - Graduation year used in assessment (correlates with age)\n"
        "   - Location or timezone preferences (may exclude demographics)\n"
        "   - Name or cultural markers influencing assessment\n\n"
        "3. **Requirement Inflation Check**:\n"
        "   - Are any requirements higher than the job actually needs?\n"
        "   - Does '3+ years' effectively exclude capable but less-experienced candidates?\n"
        "   - Does the degree requirement exclude self-taught engineers?\n\n"
        "4. **Evaluation Consistency Check**:\n"
        "   - Were criteria applied to information on the resume, not assumptions?\n"
        "   - Were gaps treated neutrally (not automatically negative)?\n"
        "   - Was the recommendation evidence-based?\n\n"
        "5. **Compliance Risk Flags**:\n"
        "   - EEOC-relevant issues\n"
        "   - ADA compliance concerns\n"
        "   - ADEA (age discrimination) risks\n"
        "   - GDPR considerations (if EU candidate)\n\n"
        "For EACH finding:\n"
        "- Description\n"
        "- Severity: Critical / Major / Minor\n"
        "- Which section is affected\n"
        "- Recommended fix\n\n"
        "Conclude with:\n"
        "- Issue count by severity\n"
        "- Verdict: PASS / CONDITIONAL PASS / FAIL\n"
        "- Mandatory disclaimer: 'This AI-powered bias review is a supplementary "
        "  check. It cannot guarantee bias-free outcomes and should not replace "
        "  human I/O psychology review or legal counsel for production hiring.'\n"
    ),
    expected_output=(
        "A bias audit report with findings across 5 dimensions (language bias, "
        "proxy discrimination, requirement inflation, evaluation consistency, "
        "compliance), severity ratings, verdict, and mandatory AI limitation disclaimer."
    ),
    agent=bias_reviewer,
)

print(f"Task created: Bias-Check Review -> {task_bias.agent.role}")

### Pipeline Visualization

In [ ]:
tasks = [task_screen, task_questions, task_summary, task_bias]
task_names = ["Resume Screening", "Interview Questions", "Candidate Summary", "Bias-Check Audit"]
phases = ["SCREEN", "PREPARE", "BRIEF", "AUDIT"]

print("HIRING PIPELINE FLOW")
print("=" * 70)
for i, (phase, name, task) in enumerate(zip(phases, task_names, tasks)):
    print(f"  [{phase}] {name}")
    print(f"    Agent: {task.agent.role.split('&')[0].strip()}")
    if i < len(tasks) - 1:
        print(f"    {'':6}│")
        print(f"    {'':6}│  output feeds next phase ↓")
        print(f"    {'':6}│")
print("=" * 70)
print("\nBias-Check Reviewer sees ALL prior outputs and audits the entire flow.")

## 5. Assemble and Run the Crew

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")

In [ ]:
print("=" * 70)
print(f"LAUNCHING HIRING PIPELINE CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Role: {JOB_DESCRIPTION['title']}")
print(f"Candidate: Alex Morgan")
print(f"Flow: SCREEN -> PREPARE -> BRIEF -> AUDIT")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — Bias-Check Audit

In [ ]:
print("=" * 70)
print("BIAS-CHECK AUDIT REPORT")
print("=" * 70)
print(result.raw)

### 6.2 Phase-by-Phase Outputs

In [ ]:
for phase, name, task in zip(phases, task_names, tasks):
    print("\n" + "=" * 70)
    print(f"[{phase}] {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2500:
            print(text[:2500])
            print(f"\n... [{len(text) - 2500} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Pipeline Metrics

In [ ]:
print("HIRING PIPELINE METRICS")
print("=" * 65)
print(f"{'Phase':<8} {'Task':<25} {'Agent':<27} {'Output':>8}")
print("-" * 65)

total = 0
for phase, name, task in zip(phases, task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    agent_short = task.agent.role.split("&")[0].strip()[:25]
    print(f"{phase:<8} {name:<25} {agent_short:<27} {length:>6,}")

print("-" * 65)
print(f"{'':8} {'TOTAL':<52} {total:>6,}")
print(f"\nRole: {JOB_DESCRIPTION['title']}")
print(f"Candidate: Alex Morgan")

if hasattr(result, 'token_usage') and result.token_usage:
    print(f"Tokens: {result.token_usage.get('total_tokens', 'N/A')}")

## 7. Save Hiring Package

In [ ]:
# Save the complete hiring package
package_lines = [
    f"# Hiring Package: {JOB_DESCRIPTION['title']}",
    f"**Candidate:** Alex Morgan",
    f"**Generated:** {datetime.now().isoformat()}",
    "**⚠️ AI-Generated — Requires Human Review Before Any Hiring Decision**",
    "---",
]

section_titles = [
    "Phase 1: Resume Screening (SCREEN)",
    "Phase 2: Interview Questions (PREPARE)",
    "Phase 3: Candidate Summary (BRIEF)",
    "Phase 4: Bias-Check Audit (AUDIT)",
]
for title, task in zip(section_titles, tasks):
    package_lines.append(f"\n## {title}\n")
    package_lines.append(task.output.raw if task.output else "[No output]")
    package_lines.append("\n---")

package = "\n".join(package_lines)
with open("hiring_package.md", "w", encoding="utf-8") as f:
    f.write(package)

print(f"Hiring package saved: hiring_package.md ({len(package):,} chars)")
print("⚠️ This is an AI-generated document for educational purposes only.")

## 8. Experiment: Different Candidate Profile

Run the same pipeline on a candidate with a **non-traditional background** to observe how the agents handle alternative career paths and whether the bias reviewer catches any unfair treatment.

In [ ]:
# =====================================================
# SECOND CANDIDATE — Non-traditional background
# =====================================================
RESUME_2 = """
TAYLOR NGUYEN
taylor.nguyen@email.com | linkedin.com/in/taylornguyen

SUMMARY
Self-taught ML practitioner transitioning from 6 years in mechanical
engineering. Built production ML systems during a career pivot, combining
domain expertise in manufacturing with modern ML techniques.

EXPERIENCE

ML Engineer — AutoML Startup (remote, 2023–present, 1.5 years)
- Built production recommendation engine serving 2M users using Python and
  PyTorch, deployed on AWS (SageMaker, Lambda, S3)
- Implemented A/B testing framework and feature store using Feast
- Reduced model training time by 60% by migrating to distributed training
- Solo ML engineer; designed architecture decisions independently

Mechanical Engineer → Data Analyst → ML Transition (2017–2023)
- Mechanical Engineer at Ford Motor Company (2017–2020, 3 years):
  Applied statistical process control and predictive maintenance models
  using MATLAB and Python for manufacturing quality analysis
- Self-study period (2020–2021, 1 year):
  Completed fast.ai courses, Kaggle competitions (silver medal x2),
  built 5 ML projects on GitHub
- Data Analyst at ManuTech Inc (2021–2023, 2 years):
  Transitioned to data role; built anomaly detection models for factory
  sensor data using scikit-learn and SQL, deployed using Docker + Flask

EDUCATION
B.S. Mechanical Engineering — Community College of Denver (2017)
fast.ai Practical Deep Learning Certificate (2021)
AWS Machine Learning Specialty Certification (2023)

SKILLS
Python, PyTorch, scikit-learn, SQL, AWS (SageMaker, Lambda, S3, EC2),
Docker, Git, Feast, pandas, numpy, MATLAB, flask, FastAPI

PROJECTS
- Kaggle: 2 silver medals (tabular prediction competitions)
- GitHub: 15 repositories, 200+ stars total on ML projects
- Blog: taylornguyen.dev — 20+ technical posts on ML engineering
"""

print("CANDIDATE 2 — Non-Traditional Background")
print("=" * 60)
print(RESUME_2)

In [ ]:
hiring_context_2 = f"{jd_text}\n=== CANDIDATE RESUME ===\n{RESUME_2}"

task2_screen = Task(
    description=(
        f"Screen this candidate:\n\n{hiring_context_2}\n\n"
        "Produce structured screening scorecard. Mark each requirement as "
        "CONFIRMED / PARTIAL / INFERRED / NOT FOUND with evidence. Note the "
        "non-traditional career path but evaluate on SKILLS, not path."
    ),
    expected_output="Screening scorecard with requirements match and fit assessment.",
    agent=resume_screener,
)

task2_questions = Task(
    description=(
        f"Design 12-15 interview questions for this candidate.\n"
        f"Role: {JOB_DESCRIPTION['title']}\n\n"
        "NOTE: This candidate has a non-traditional background (mechanical "
        "engineering → self-taught ML). Design questions that FAIRLY assess "
        "their actual ML skills without penalizing their career path. "
        "Avoid questions that only someone with a CS degree could answer."
    ),
    expected_output="12-15 interview questions across 4 categories.",
    agent=question_generator,
)

task2_summary = Task(
    description=(
        f"One-page candidate brief for the hiring committee.\n"
        f"Role: {JOB_DESCRIPTION['title']}\n"
        "Candidate: Taylor Nguyen\n\n"
        "Include: snapshot, strengths, concerns, interview focus, recommendation "
        "with confidence, and AI Assessment Limitations section."
    ),
    expected_output="Candidate brief with recommendation and limitations section.",
    agent=candidate_summarizer,
)

task2_bias = Task(
    description=(
        "Audit the hiring package for Taylor Nguyen for bias.\n\n"
        "PAY SPECIAL ATTENTION to bias against non-traditional backgrounds:\n"
        "- Was the community college degree treated differently than a top-tier "
        "  university would be? (educational pedigree bias)\n"
        "- Was the self-study gap treated as a negative? (career gap bias)\n"
        "- Was the mechanical engineering background treated as irrelevant "
        "  rather than complementary? (career change bias)\n"
        "- Were Kaggle accomplishments and open-source work given appropriate "
        "  weight vs. traditional credentials?\n\n"
        "Audit all 5 dimensions. Include mandatory AI limitation disclaimer."
    ),
    expected_output="Bias audit with special attention to non-traditional background bias.",
    agent=bias_reviewer,
)

tasks_2 = [task2_screen, task2_questions, task2_summary, task2_bias]
print(f"Candidate 2 tasks created: {len(tasks_2)}")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Candidate: Taylor Nguyen (non-traditional background)")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print("BIAS AUDIT — Taylor Nguyen (Non-Traditional)")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Candidates

In [ ]:
print("PIPELINE COMPARISON — Traditional vs. Non-Traditional Candidate")
print("=" * 72)
print(f"{'Phase':<8} {'Task':<25} {'Alex Morgan':>14} {'Taylor Nguyen':>14}")
print("-" * 72)

for phase, name, t1, t2 in zip(phases, task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{phase:<8} {name:<25} {len1:>11,} ch {len2:>11,} ch")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 72)
print(f"{'':8} {'TOTAL':<25} {total1:>11,} ch {total2:>11,} ch")

print(f"\nAlex Morgan: CS degree, linear career, traditional path")
print(f"Taylor Nguyen: MechE degree, self-taught pivot, community college")
print(f"\n⚠️ Compare outputs for differential treatment of non-traditional paths.")

## 10. Advanced: Parallel Screening with Bias-First Design

### Alternative Pattern: Bias Review Before Recommendation

In the standard pipeline, the bias reviewer audits AFTER the recommendation is made. An alternative is to check for bias BEFORE the candidate summary:

```
               ┌──> Question Generator ──┐
Screener ──────┤                         ├──> Candidate Summary (bias-aware)
               └──> Bias Reviewer ───────┘
```

The candidate summarizer sees both the questions AND the bias audit, allowing it to write recommendations that already account for flagged issues.

In [ ]:
# Bias-first pattern with explicit context

bf_screen = Task(
    description=(
        f"Screen this candidate:\n\n{hiring_context}\n\n"
        "Produce structured scorecard with CONFIRMED/PARTIAL/INFERRED/NOT FOUND."
    ),
    expected_output="Screening scorecard.",
    agent=resume_screener,
)

bf_questions = Task(
    description=(
        f"Design 12-15 interview questions for: {JOB_DESCRIPTION['title']}\n"
        "Based on screening results. Include technical, gap, behavioral, scenario."
    ),
    expected_output="Interview question set.",
    agent=question_generator,
    context=[bf_screen],  # Sees screening only
)

# Bias reviewer runs BEFORE the final summary
bf_bias = Task(
    description=(
        "Audit the screening scorecard and interview questions for bias. "
        "Check language, proxy discrimination, requirement inflation, and "
        "evaluation consistency. This audit runs BEFORE the recommendation."
    ),
    expected_output="Pre-recommendation bias audit.",
    agent=bias_reviewer,
    context=[bf_screen, bf_questions],  # Sees screening + questions
)

# Candidate summary sees ALL three — including the bias audit
bf_summary = Task(
    description=(
        f"Write candidate brief for: {JOB_DESCRIPTION['title']}\n"
        "Candidate: Alex Morgan\n\n"
        "IMPORTANT: You have received a BIAS AUDIT alongside the screening and "
        "questions. Address any bias findings in your recommendation. If the bias "
        "reviewer flagged issues, acknowledge them and adjust your assessment."
    ),
    expected_output="Bias-aware candidate brief.",
    agent=candidate_summarizer,
    context=[bf_screen, bf_questions, bf_bias],  # Sees everything including bias
)

bf_tasks = [bf_screen, bf_questions, bf_bias, bf_summary]

print("BIAS-FIRST PIPELINE")
print("=" * 60)
print("              ┌──> Questions    ──┐")
print("  Screener ───┤                   ├──> Bias-Aware Summary")
print("              └──> Bias Reviewer ─┘")
print()
bf_names = ["Screening", "Questions", "Bias Audit (pre-rec)", "Summary (bias-aware)"]
for i, (name, task) in enumerate(zip(bf_names, bf_tasks), 1):
    ctx = task.context
    if ctx is None:
        ctx_str = "auto (all prior)"
    elif len(ctx) == 0:
        ctx_str = "independent"
    else:
        ctx_str = " + ".join(t.agent.role.split("&")[0].strip()[:22] for t in ctx)
    print(f"  Task {i}: {name}")
    print(f"    Context: {ctx_str}")

In [ ]:
crew_bf = Crew(
    agents=agents,
    tasks=bf_tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching bias-first crew — {datetime.now().strftime('%H:%M:%S')}")
result_bf = crew_bf.kickoff()
print(f"\nBias-first crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("BIAS-FIRST PIPELINE RESULTS")
print("=" * 65)
for name, task in zip(bf_names, bf_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx = task.context
    if ctx is None:
        ctx_str = "auto"
    elif len(ctx) == 0:
        ctx_str = "independent"
    else:
        ctx_str = f"{len(ctx)} inputs"
    print(f"  {name:<30} {length:>6,} chars | Context: {ctx_str}")

## 11. Limitations — Deep Dive

### The "Auditing Yourself" Problem

The bias-check reviewer uses the **same LLM** that produced the potentially biased outputs. This creates a fundamental limitation:

```
Same Model
    │
    ├──> Produces screening output (may contain bias)
    │
    └──> Audits screening output (may not detect its own biases)
```

**Why this matters:** LLMs have systematic biases embedded in their training data. If the model has learned that "strong" engineers went to top-10 universities, both the screener and the bias reviewer share that bias — the reviewer is unlikely to flag it.

**Mitigation strategies (not implemented here):**
1. Use a DIFFERENT model for bias review than for screening
2. Use rule-based checks (regex for gendered language, name lists for ethnicity detection) alongside LLM review
3. Pair AI review with human I/O psychologist review
4. Audit outcomes over many candidates for disparate impact (80% rule)

### The Hallucination Problem in Screening

LLMs frequently **infer skills not stated on the resume**:
- Seeing "Data Scientist" → inferring Python proficiency (usually correct, but not always)
- Seeing "ML Engineer at Google" → inferring distributed systems skills (assumption)
- Seeing "NLP models" → inferring PyTorch experience (could have used TensorFlow)

Our mitigation (CONFIRMED vs. INFERRED marking) helps but relies on the LLM following the instruction, which is not guaranteed.

### The Consistency Problem

The same resume processed twice may produce different screening scores because:
- LLM generation is stochastic (temperature > 0)
- The model may emphasize different resume sections each time
- Score calibration drifts across different candidates

**In production:** Use temperature=0, run each resume through the pipeline multiple times, and flag candidates where scores vary significantly.

### The Legal Compliance Gap

No LLM can guarantee compliance with:
- **EEOC** guidelines on adverse impact
- **ADA** requirements for reasonable accommodation
- **ADEA** protections against age discrimination
- **Local laws** that vary by jurisdiction (ban-the-box, salary history bans, etc.)

**Bottom line:** AI in hiring is a decision-support tool, not a decision-making tool. Every output in this pipeline should be reviewed by a qualified human before influencing a hiring decision.

## 12. Key Takeaways

### What We Built
- A **4-agent hiring pipeline** (Screen → Prepare → Brief → Audit) with an embedded bias-check reviewer
- Processed **two candidates** with different backgrounds (traditional CS + non-traditional career pivot) to test fairness
- Demonstrated **bias-first pipeline design** where the bias audit runs before the recommendation

### Limitations Awareness
1. **Same-model auditing** — The bias reviewer shares the LLM's own biases and cannot detect what it doesn't perceive as bias
2. **Hallucinated qualifications** — LLMs infer skills not on the resume; CONFIRMED/INFERRED marking mitigates but doesn't solve
3. **Inconsistent scoring** — Stochastic generation means the same resume can receive different scores
4. **No legal compliance** — LLMs cannot guarantee EEOC, ADA, ADEA, or local employment law compliance
5. **Proxy discrimination** — University names, zip codes, and graduation years can serve as proxies for protected characteristics that the model may not flag

### Agent Design Lessons
- **Resume Screener:** Require CONFIRMED vs. INFERRED labeling — it forces the model to acknowledge uncertainty
- **Question Generator:** Explicitly list forbidden topics (protected characteristics) — LLMs don't automatically know employment law
- **Candidate Summarizer:** Mandate an "AI Limitations" section — it signals to human reviewers where to apply their judgment
- **Bias Reviewer:** Acknowledge its own limitation in the backstory — "I cannot guarantee bias-free outcomes" prevents false confidence

### Production Safeguards (Not Implemented Here)
- Use different models for screening vs. bias review
- Add rule-based bias checks alongside LLM review
- Track outcomes across demographics for disparate impact analysis
- Require human review for every reject decision
- Log all AI-generated assessments for legal auditability
- Set temperature=0 and run multiple passes for consistency